<a href="https://colab.research.google.com/github/Hem1144/ml-codes/blob/regression/regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
#initialize SparkSession and installed Required Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()


spark


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = spark.read.csv("/content/drive/MyDrive/ml/Diabetes.csv", header=True, inferSchema=True)

df.show()

print("Total numbers of records", df.count())
print("Total numbers of columns", len(df.columns))
print("Total Partition", df.rdd.getNumPartitions())

+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+
| id|age|   sex|  bmi|    bp|    tc|  ldl|  hdl| tch|  ltg|   glu|progression|
+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+
|  0| 22|  Male|20.33|106.25|140.89|-0.45|62.83|0.77|-0.54|130.98|      29.83|
|  1| 41|Female|25.99|132.23|129.64|-1.11|37.26|0.81| 1.64|151.19|      46.66|
|  2| 51|Female|32.76| 127.0|220.36|-1.69|49.56|0.41|-0.88|176.22|      59.97|
|  3| 26|  Male|35.87| 138.4|194.19|-0.04|55.57|0.45|-1.38|125.32|      42.44|
|  4| 42|Female| 21.5|122.33|275.79| 1.19|63.64|0.54|-0.69|184.72|      49.36|
|  5| 47|  Male|31.62|137.18|232.35|-1.65|36.68|0.26| 1.63| 99.83|      54.15|
|  6| 22|  Male|37.06|106.47|244.34|-0.07|34.22|0.48|-1.72| 65.41|      39.55|
|  7| 23|  Male|26.75|129.39|177.69|-0.37|42.03|0.68| 0.82|180.71|      33.12|
|  8| 44|Female|28.38|125.23|276.99| 0.69|55.18| 1.1|-0.52|134.71|      53.84|
|  9| 36|  Male|36.83| 121.9|174.81|  1.1|53.55|0.25

In [7]:
# show Schema,Prints the structure of the dataset
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- bp: double (nullable = true)
 |-- tc: double (nullable = true)
 |-- ldl: double (nullable = true)
 |-- hdl: double (nullable = true)
 |-- tch: double (nullable = true)
 |-- ltg: double (nullable = true)
 |-- glu: double (nullable = true)
 |-- progression: double (nullable = true)



In [8]:
#Statistical Analysis
df.describe().show()

+-------+-----------------+------------------+-------+------------------+------------------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-----------------+
|summary|               id|               age|    sex|               bmi|                bp|                tc|                 ldl|              hdl|                tch|                 ltg|               glu|      progression|
+-------+-----------------+------------------+-------+------------------+------------------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-----------------+
|  count|          3045570|           3045570|3045570|           3045570|           3045570|           3045570|             3045570|          3045570|            3045570|             3045570|           3045570|          3045570|
|   mean|        1522784.5| 39.50258309610352|   NULL|30.005921735504103|120.0061583